# Visvesvaraya Technological University Belagavi-590 014
## Lingaraj Appa Engineering College, Bidar - 585403 (2026-27)
### Department of Computer Science & Engineering

**Project Title:** Machine Learning–Based Dynamic Pricing Model for E-Commerce  
**Syllabus Ref:** `DPS_fb375022.pdf`  
**Student Authors:** Channaveer (3LA22CS014), Abhishek (3LA23CS005), Avishkar (3LA23CS019), Pavan (3LA23CS060)  
**Faculty Mentors:** Dr. Sripal Reddy (Project Guide), Dr. Shilpa Harnale (Project Coordinator)  

---

## Phase 1: Problem Statement & Objectives
Static e-commerce pricing leads to either overpricing (customer churn) or underpricing (eroded margins). This machine learning dynamic pricing system analyzes market signals—demand volume, stock depletion, and competitor pricing—to predict the optimal selling price using Linear Regression, Decision Tree Regressor, and Random Forest Regressor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

print("All core libraries successfully imported!")

## Phase 2: Data Collection & Preparation

In [ ]:
df = pd.read_csv('ecommerce_pricing_dataset.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Phase 3: Feature Engineering & Exploratory Data Analysis (EDA)
- `demand_to_stock_ratio` = demand / (stock + 1)
- `competitor_price_diff` = competitor_price - current_price
- `season_intensity` = encoded seasonal factor

In [ ]:
df['demand_to_stock_ratio'] = df['demand'] / (df['stock_available'] + 1.0)
df['competitor_price_diff'] = df['competitor_price'] - df['current_price']
df['competitor_price_ratio'] = df['competitor_price'] / df['current_price']
df['historical_price_change'] = (df['current_price'] - df['historical_price']) / df['historical_price']

season_map = {'Festival': 1.30, 'Peak': 1.15, 'Normal': 1.00, 'Clearance': 0.80}
df['season_intensity'] = df['season'].map(season_map).fillna(1.0)

cat_dummies = pd.get_dummies(df['category'], prefix='cat', drop_first=True, dtype=float)
df = pd.concat([df, cat_dummies], axis=1)

print("Feature engineering completed!")

## Phase 4: Model Training (80/20 Train-Test Split)

In [ ]:
feature_cols = [
    'current_price', 'historical_price', 'demand', 'stock_available',
    'competitor_price', 'units_sold', 'demand_to_stock_ratio',
    'competitor_price_diff', 'competitor_price_ratio', 'historical_price_change',
    'season_intensity', 'promotion_active'
] + [c for c in df.columns if c.startswith('cat_')]

X = df[feature_cols].values
y = df['target_price'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Linear Regression': (Ridge(alpha=1.0), True),
    'Decision Tree': (DecisionTreeRegressor(max_depth=5, min_samples_leaf=3, random_state=42), False),
    'Random Forest': (RandomForestRegressor(n_estimators=30, max_depth=6, min_samples_leaf=2, random_state=42), False)
}

benchmark = []
for name, (model, req_scale) in models.items():
    if req_scale:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    benchmark.append({'Model': name, 'MAE (INR)': round(mae, 2), 'RMSE (INR)': round(rmse, 2), 'R2 Score': round(r2, 4)})

pd.DataFrame(benchmark)

## Phase 5 & 6: Price Prediction Engine with Guardrails

In [ ]:
best_model = models['Random Forest'][0]

sample_input = np.array([X_test[0]])
predicted_price = float(best_model.predict(sample_input)[0])
actual_target = y_test[0]

print(f"Predicted Optimal Price: INR {predicted_price:.2f}")
print(f"Actual Target Price:    INR {actual_target:.2f}")
print(f"Prediction Error:       INR {abs(predicted_price - actual_target):.2f}")